# 🛡️ CNN Phân Loại Mã Độc — Pure INT8 QAT

**Kiến trúc:** CNN (8×32×1 input)  
**Quantization:** Quantization-Aware Training — Pure INT8 (Chi-You style)  
**Mục tiêu:** Xuất weights INT8 thuần túy để nạp thẳng lên phần cứng (FPGA/ASIC)  

---

### Luồng xử lý
```
Dataset ảnh PNG
    ↓ (chỉ lần đầu — lần sau load cache)
Tiền xử lý → cache dataset.npz  
    ↓
Train float32  (10 epochs)
    ↓
QAT fine-tune  (5 epochs) — weights/activations bị ép vào lưới INT8
    ↓
Export: weights INT8 + shift_params → JSON / Excel
```

### Quy ước INT8 trong notebook này
| Vị trí | Range | Ghi chú |
|---|---|---|
| Weights (Conv, Dense) | [-127, 127] | Signed INT8 |
| Activations (ReLU) | [0, 127] | Unsigned INT8 |
| Accumulator MAC | INT32 | Tránh overflow |
| Output Dense(2) | INT32 thô | So sánh trực tiếp, không cần softmax |

> ⚠️ Bật GPU trước khi chạy: `Runtime → Change runtime type → T4 GPU`

## 📦 1. Cài Đặt Thư Viện

In [ ]:
!pip install -q openpyxl
print('✅ Cài đặt xong!')

## ☁️ 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mount thành công!')

## ⚙️ 3. Cấu Hình Tham Số

In [ ]:
# ================================================================
#  THAY ĐỔI CÁC GIÁ TRỊ NÀY TRƯỚC KHI CHẠY
# ================================================================

# Thư mục chứa dataset ảnh PNG
# Cấu trúc:
#   malimg/
#     benign/          ← file .png benign
#     Allaple.A/       ← file .png malware family
#     Allaple.L/
#     ...
DATASET_PATH = '/content/drive/MyDrive/CNX/malimg'

# Thư mục lưu toàn bộ kết quả (model, weights, cache, biểu đồ...)
OUTPUT_DIR   = '/content/drive/MyDrive/CNX'

# Random seed — đồng bộ hóa toàn bộ pipeline
RANDOM_SEED  = 42

# ── Siêu tham số huấn luyện ──────────────────────────────────────
BATCH_SIZE   = 32
EPOCHS_F32   = 10    # Giai đoạn 1: Train float32 bình thường
EPOCHS_QAT   = 5     # Giai đoạn 2: QAT fine-tune
TEST_SPLIT   = 0.3

# ── Tham số Quantization ─────────────────────────────────────────
# SHIFT_BITS: số bit dịch phải sau MAC (INT32 → INT8)
# Công thức tham khảo: SHIFT_BITS = 7 cho mạng 2 conv layer
# Tăng lên 8 nếu thấy activation bị bão hòa (toàn 127)
SHIFT_BITS   = 7

# ── Đường dẫn cache ──────────────────────────────────────────────
import os
CACHE_PATH   = os.path.join(OUTPUT_DIR, 'cache', 'dataset.npz')

# ================================================================
print('=' * 50)
print(f'📂 Dataset     : {DATASET_PATH}')
print(f'📁 Output      : {OUTPUT_DIR}')
print(f'💾 Cache       : {CACHE_PATH}')
print(f'🎲 Seed        : {RANDOM_SEED}')
print(f'🔁 Epochs F32  : {EPOCHS_F32}')
print(f'🔁 Epochs QAT  : {EPOCHS_QAT}')
print(f'⚙️  Shift bits  : {SHIFT_BITS}')
print('=' * 50)

## 📚 4. Import Thư Viện

In [ ]:
import os, glob, random, time, json
import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Flatten, Conv2D, MaxPooling2D, Layer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl

# Tạo thư mục cần thiết
for d in [OUTPUT_DIR,
          os.path.join(OUTPUT_DIR, 'cache'),
          os.path.join(OUTPUT_DIR, 'dat'),
          os.path.join(OUTPUT_DIR, 'hardware')]:
    os.makedirs(d, exist_ok=True)

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

## 📂 5. Đọc & Tiền Xử Lý Dữ Liệu (Có Cache)

**Logic cache:**
- Nếu `dataset.npz` đã tồn tại → load thẳng, **bỏ qua toàn bộ bước đọc ảnh**
- Nếu chưa có → đọc ảnh, cân bằng, chuẩn hóa, **lưu cache** để lần sau dùng lại

In [ ]:
if os.path.exists(CACHE_PATH):
    # ── LOAD CACHE ───────────────────────────────────────────────
    print(f'✅ Tìm thấy cache tại: {CACHE_PATH}')
    print('   Đang load cache — bỏ qua bước đọc ảnh...')
    cache = np.load(CACHE_PATH)
    X_train = cache['X_train']
    X_test  = cache['X_test']
    y_train = cache['y_train']
    y_test  = cache['y_test']
    print(f'   X_train : {X_train.shape} | y_train : {y_train.shape}')
    print(f'   X_test  : {X_test.shape}  | y_test  : {y_test.shape}')
    print('✅ Load cache hoàn tất!')

else:
    # ── BUILD FROM SCRATCH ───────────────────────────────────────
    print('⚠️  Không tìm thấy cache — Đang đọc ảnh từ dataset...')

    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(f'❌ Không tìm thấy dataset tại: {DATASET_PATH}')

    benign_paths  = []
    malware_paths = []

    for fam_name in os.listdir(DATASET_PATH):
        fam_dir = os.path.join(DATASET_PATH, fam_name)
        if not os.path.isdir(fam_dir):
            continue
        imgs = glob.glob(os.path.join(fam_dir, '*.png'))
        if fam_name.lower() == 'benign':
            benign_paths.extend(imgs)
        else:
            malware_paths.extend(imgs)

    # Cân bằng: Malware = 2 × Benign
    num_benign     = len(benign_paths)
    target_malware = num_benign * 2
    print(f'   Benign gốc  : {num_benign}')
    print(f'   Malware gốc : {len(malware_paths)}')

    if len(malware_paths) > target_malware:
        random.seed(RANDOM_SEED)
        malware_paths = random.sample(malware_paths, target_malware)
    print(f'   Malware sau downsampling: {len(malware_paths)}')

    # Đọc ảnh → resize 8×32 → float32 [0,1]
    X, y = [], []
    print('   Đang đọc ảnh Benign...')
    for p in benign_paths:
        with Image.open(p) as im:
            X.append(np.array(im.resize((8, 32), Image.Resampling.LANCZOS)))
            y.append([1, 0])

    print('   Đang đọc ảnh Malware...')
    for p in malware_paths:
        with Image.open(p) as im:
            X.append(np.array(im.resize((8, 32), Image.Resampling.LANCZOS)))
            y.append([0, 1])

    X = np.array(X).astype('float32') / 255.0
    y = np.array(y)
    total = len(X)
    X = X.reshape(total, 8, 32, 1)

    # Train/Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SPLIT, random_state=RANDOM_SEED
    )

    # Lưu cache
    np.savez_compressed(CACHE_PATH,
                        X_train=X_train, X_test=X_test,
                        y_train=y_train, y_test=y_test)
    print(f'✅ Đã lưu cache tại: {CACHE_PATH}')

# Thống kê phân phối
print('\n' + '-'*45)
print('THỐNG KÊ PHÂN PHỐI DỮ LIỆU')
print('-'*45)
print(f'TỔNG  : Benign={int(np.sum(np.concatenate([y_train,y_test])[:,0]))} | Malware={int(np.sum(np.concatenate([y_train,y_test])[:,1]))}')
print(f'TRAIN : Benign={int(np.sum(y_train[:,0]))} | Malware={int(np.sum(y_train[:,1]))}')
print(f'TEST  : Benign={int(np.sum(y_test[:,0]))}  | Malware={int(np.sum(y_test[:,1]))}')
print('-'*45)

## 🧮 6. Định Nghĩa Các Lớp QAT (Fake-Quantize + STE)

**Straight-Through Estimator (STE):** Hàm `round()` và `clip()` không có gradient.  
STE cho phép gradient **đi thẳng qua** trong backward pass — mạng vẫn học được.

```
Forward:   output = round(clip(x, -127, 127))
Backward:  gradient đi qua như identity (∂output/∂x = 1)
```

In [ ]:
# ── Hàm STE cốt lõi ──────────────────────────────────────────────
def ste_round(x):
    """Round với gradient thẳng (STE)."""
    return x + tf.stop_gradient(tf.round(x) - x)


def fake_quant_weight(w, n_bits=8):
    """
    Fake-quantize weight về lưới INT8 [-127, 127].
    - Forward : w bị ép vào lưới int — mô phỏng sai số lượng tử hóa
    - Backward: gradient đi qua bình thường (STE)
    Kết quả vẫn là float32 để Keras tính loss, nhưng chỉ nhận
    các giá trị nguyên trong [-127, 127].
    """
    q_max = 2 ** (n_bits - 1) - 1          # = 127
    # Scale weight về [-127, 127] dựa theo giá trị tuyệt đối lớn nhất
    scale = tf.reduce_max(tf.abs(w)) / q_max + 1e-8
    w_scaled = w / scale
    w_clamp  = tf.clip_by_value(w_scaled, -q_max, q_max)
    w_round  = ste_round(w_clamp)           # INT8 trên lưới, gradient vẫn chạy
    return w_round * scale                  # trả về float scale để loss đúng


def fake_quant_activation(x):
    """
    Fake-quantize activation sau ReLU về [0, 127].
    ReLU đảm bảo x >= 0, chỉ cần clamp trên là 127.
    """
    x_clamp = tf.clip_by_value(x, 0.0, 127.0)
    return ste_round(x_clamp)


# ── Custom layer QAT-aware ────────────────────────────────────────
class QATConv2D(Layer):
    """
    Conv2D với fake-quantize weight trong forward pass.
    QUAN TRỌNG: fake_quant_weight() CHỈ được dùng để tính output tạm thời.
    Tuyệt đối KHÔNG .assign() lên kernel thật — optimizer phải thấy
    weight float32 gốc để cập nhật gradient đúng cách.
    """
    def __init__(self, filters, kernel_size, strides=(1,1),
                 padding='valid', name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.filters     = filters
        self.kernel_size = kernel_size if isinstance(kernel_size, (list,tuple)) else (kernel_size, kernel_size)
        self.strides     = strides
        self.pad         = padding.upper()
        # Dùng Conv2D nội bộ chỉ để khởi tạo và giữ weight/bias
        self.conv = Conv2D(filters, kernel_size, strides=strides,
                           padding=padding, activation=None,
                           use_bias=True,
                           name=f'{name}_conv' if name else None)

    def call(self, x, training=False):
        if training:
            # Fake-quant weight TẠM THỜI để tính output
            # Weight thật (self.conv.kernel) KHÔNG bị thay đổi
            w_fq = fake_quant_weight(self.conv.kernel)   # float32, nhưng trên lưới INT8
            out  = tf.nn.conv2d(
                x, w_fq,
                strides=[1, self.strides[0], self.strides[1], 1],
                padding=self.pad
            ) + self.conv.bias
        else:
            # Inference: dùng weight thật float32 bình thường
            out = self.conv(x)

        # ReLU luôn áp dụng
        out = tf.nn.relu(out)

        # Fake-quant activation khi train: ép về lưới [0, 127]
        if training:
            out = fake_quant_activation(out)
        return out

    def get_int8_weights(self):
        """Sau QAT xong: round weight thật về INT8 numpy để nạp lên hardware."""
        w = self.conv.kernel.numpy()
        b = self.conv.bias.numpy()
        q_max = 127
        scale_w = np.max(np.abs(w)) / q_max + 1e-8
        scale_b = np.max(np.abs(b)) / q_max + 1e-8 if np.max(np.abs(b)) > 0 else 1.0
        w_int8  = np.clip(np.round(w / scale_w), -q_max, q_max).astype(np.int8)
        b_int8  = np.clip(np.round(b / scale_b), -q_max, q_max).astype(np.int8)
        return w_int8, b_int8


class QATDense(Layer):
    """
    Dense với fake-quantize weight.
    - Các layer ẩn (use_relu=True): ReLU + fake-quant activation [0, 127]
    - Layer output (use_relu=False): KHÔNG ReLU, KHÔNG quant activation
      → giữ raw accumulator để hardware so sánh out[0] vs out[1]
    """
    def __init__(self, units, activation=None, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.dense    = Dense(units, activation=None, use_bias=True,
                              name=f'{name}_dense' if name else None)
        self.use_relu = (activation == 'relu')

    def call(self, x, training=False):
        if training:
            # Fake-quant weight TẠM THỜI — KHÔNG .assign() lên kernel thật
            w_fq = fake_quant_weight(self.dense.kernel)
            out  = tf.matmul(x, w_fq) + self.dense.bias
        else:
            out = self.dense(x)

        if self.use_relu:
            out = tf.nn.relu(out)
            if training:
                out = fake_quant_activation(out)
        return out

    def get_int8_weights(self):
        """Sau QAT xong: round weight thật về INT8 numpy để nạp lên hardware."""
        w = self.dense.kernel.numpy()
        b = self.dense.bias.numpy()
        q_max = 127
        scale_w = np.max(np.abs(w)) / q_max + 1e-8
        scale_b = np.max(np.abs(b)) / q_max + 1e-8 if np.max(np.abs(b)) > 0 else 1.0
        w_int8  = np.clip(np.round(w / scale_w), -q_max, q_max).astype(np.int8)
        b_int8  = np.clip(np.round(b / scale_b), -q_max, q_max).astype(np.int8)
        return w_int8, b_int8


print('✅ Định nghĩa các lớp QAT (STE + fake-quant) hoàn tất!')

## 🧠 7. Xây Dựng Mô Hình QAT

Kiến trúc giữ nguyên như mô hình gốc, chỉ thay Conv2D/Dense thường  
bằng **QATConv2D / QATDense** đã tích hợp fake-quantize.

```
Input (8×32×1)
    → QATConv2D(16, 3×3)  →  ReLU  →  fake_quant [0,127]
    → MaxPooling2D(2×2)
    → QATConv2D(32, 3×3)  →  ReLU  →  fake_quant [0,127]
    → Flatten
    → QATDense(48, relu)  →  ReLU  →  fake_quant [0,127]
    → QATDense(2)         →  (không ReLU, giữ INT32 thô)
```

In [ ]:
def build_qat_model(input_shape=(8, 32, 1), num_classes=2):
    inp = keras.Input(shape=input_shape, name='Input')

    x = QATConv2D(16, (3,3), strides=(1,1), padding='valid', name='Conv2D_1')(inp)
    x = MaxPooling2D(pool_size=(2,2), name='MaxPooling_1')(x)
    x = QATConv2D(32, (3,3), strides=(1,1), padding='valid', name='Conv2D_2')(x)
    x = Flatten(name='Flatten')(x)
    x = QATDense(48, activation='relu', name='Dense_1')(x)
    # Layer cuối: không ReLU, không fake-quant activation
    # → giữ raw accumulator INT32 → so sánh out[0] vs out[1]
    out = QATDense(num_classes, activation=None, name='Dense_output')(x)

    model = Model(inputs=inp, outputs=out, name='CNN_QAT_INT8')
    return model


qat_model = build_qat_model()
qat_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
qat_model.summary()

## 🏋️ 8. Giai Đoạn 1 — Train Float32

Train bình thường trước để mạng hội tụ.  
Fake-quant chỉ kích hoạt trong **giai đoạn QAT fine-tune** (bước 9).

In [ ]:
print('=' * 55)
print('GIAI ĐOẠN 1: TRAIN FLOAT32')
print('=' * 55)

tic = time.time()
history_f32 = qat_model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_F32,
    verbose=1,
    validation_split=0.2
)
toc = time.time()
print(f'\n⏱️  Thời gian train float32: {toc - tic:.2f}s')

# Lưu checkpoint float32
f32_path = os.path.join(OUTPUT_DIR, 'checkpoint_f32.weights.h5')
qat_model.save_weights(f32_path)
print(f'💾 Checkpoint float32 lưu tại: {f32_path}')

## ⚡ 9. Giai Đoạn 2 — QAT Fine-Tune

Giảm learning rate xuống 10× để fine-tune ổn định.  
Lần này `training=True` → fake-quant kích hoạt → mạng học cách  
chịu đựng sai số lượng tử hóa INT8.

In [ ]:
print('=' * 55)
print('GIAI ĐOẠN 2: QAT FINE-TUNE (fake-quant active)')
print('=' * 55)

# Giảm LR để fine-tune ổn định
qat_model.compile(
    loss='categorical_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=['accuracy']
)

tic = time.time()
history_qat = qat_model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS_QAT,
    verbose=1,
    validation_split=0.2
)
toc = time.time()
print(f'\n⏱️  Thời gian QAT fine-tune: {toc - tic:.2f}s')

# Lưu checkpoint QAT
qat_path = os.path.join(OUTPUT_DIR, 'checkpoint_qat.weights.h5')
qat_model.save_weights(qat_path)
print(f'💾 Checkpoint QAT lưu tại: {qat_path}')

## 📈 10. Biểu Đồ Quá Trình Huấn Luyện

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Quá Trình Huấn Luyện CNN QAT INT8', fontsize=14, fontweight='bold')

# Float32 accuracy
axes[0,0].plot(history_f32.history['accuracy'],     label='Train')
axes[0,0].plot(history_f32.history['val_accuracy'], label='Val')
axes[0,0].set_title('Giai đoạn 1 — Float32 Accuracy')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Accuracy')
axes[0,0].legend(); axes[0,0].grid(True)

# Float32 loss
axes[0,1].plot(history_f32.history['loss'],     label='Train')
axes[0,1].plot(history_f32.history['val_loss'], label='Val')
axes[0,1].set_title('Giai đoạn 1 — Float32 Loss')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Loss')
axes[0,1].legend(); axes[0,1].grid(True)

# QAT accuracy
axes[1,0].plot(history_qat.history['accuracy'],     label='Train', color='green')
axes[1,0].plot(history_qat.history['val_accuracy'], label='Val',   color='orange')
axes[1,0].set_title('Giai đoạn 2 — QAT Fine-tune Accuracy')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Accuracy')
axes[1,0].legend(); axes[1,0].grid(True)

# QAT loss
axes[1,1].plot(history_qat.history['loss'],     label='Train', color='green')
axes[1,1].plot(history_qat.history['val_loss'], label='Val',   color='orange')
axes[1,1].set_title('Giai đoạn 2 — QAT Fine-tune Loss')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Loss')
axes[1,1].legend(); axes[1,1].grid(True)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'training_history.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'📊 Biểu đồ lưu tại: {plot_path}')

## 🔍 11. Đánh Giá Mô Hình

In [ ]:
tic = time.time()
y_pred_raw = qat_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
toc = time.time()
print(f'⏱️  Thời gian inference: {toc - tic:.4f}s')

test_loss, test_acc = qat_model.evaluate(X_test, y_test, verbose=0)
print(f'\n📌 Test Accuracy : {test_acc:.4f}')
print(f'📌 Test Loss     : {test_loss:.4f}')

y_true = y_test.argmax(axis=1)
y_pred = y_pred_raw.argmax(axis=1)   # argmax thay softmax — đúng với hardware
target_names = ['Benign', 'Malware']

print('\n' + '='*55)
print('CLASSIFICATION REPORT')
print('='*55)
print(classification_report(y_true, y_pred, target_names=target_names))

## 🔥 12. Confusion Matrix

In [ ]:
conf_mat      = confusion_matrix(y_true, y_pred)
conf_mat_norm = np.around(
    conf_mat.astype('float') / conf_mat.sum(axis=1)[:, np.newaxis], 2
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(conf_mat,      annot=True, fmt='d',    cmap='Oranges',
            xticklabels=target_names, yticklabels=target_names, ax=ax1)
ax1.set_title('Raw Confusion Matrix'); ax1.set_ylabel('True'); ax1.set_xlabel('Predicted')

sns.heatmap(conf_mat_norm, annot=True, fmt='.2f',  cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=ax2)
ax2.set_title('Normalized Confusion Matrix'); ax2.set_ylabel('True'); ax2.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

# Lưu .dat
dat_path = os.path.join(OUTPUT_DIR, 'dat', 'conf_mat_qat_int8.dat')
with open(dat_path, 'wb') as f:
    for line in np.matrix(conf_mat_norm):
        np.savetxt(f, line, fmt='%.2f')
print(f'💾 .dat lưu tại: {dat_path}')

## 💾 13. Export Weights INT8 cho Hardware

Sau khi QAT xong, extract weights thực sự về **numpy int8**  
— đây là giá trị sẽ được nạp thẳng vào ROM/BRAM trên phần cứng.

**Kèm theo `shift_params.json`** — chứa `SHIFT_BITS` cho từng layer  
để hardware biết dịch phải bao nhiêu bit sau khi tính MAC.

```
Hardware inference flow:
  acc_int32 = MAC(input_int8, weight_int8)
  acc_int8  = clip(acc_int32 >> SHIFT_BITS, -127, 127)
  activation= clip(acc_int8, 0, 127)   ← ReLU layers only
```

In [ ]:
def extract_all_int8_weights(model):
    """
    Duyệt qua tất cả layer QAT, lấy weights đã được
    round và cast về numpy int8 thực sự.
    Trả về dict: { layer_name: { 'weights': int8[], 'biases': int8[] } }
    """
    result = {}
    for layer in model.layers:
        if isinstance(layer, (QATConv2D, QATDense)):
            w_int8, b_int8 = layer.get_int8_weights()
            result[layer.name] = {
                'weights' : w_int8,
                'biases'  : b_int8,
                'w_shape' : list(w_int8.shape),
                'b_shape' : list(b_int8.shape),
            }
            print(f'  {layer.name:20s} | W: {w_int8.shape} int8 '
                  f'| B: {b_int8.shape} int8 '
                  f'| W range: [{w_int8.min()}, {w_int8.max()}]')
    return result


print('📤 Extracting INT8 weights...')
int8_data = extract_all_int8_weights(qat_model)
print(f'\n✅ Extracted {len(int8_data)} layers')

In [ ]:
# ── Lưu Weights INT8 → JSON ───────────────────────────────────────
def save_int8_weights_json(int8_data, filepath):
    """Lưu weights INT8 dạng JSON — int8 array serialized sang list."""
    export = {}
    for name, data in int8_data.items():
        export[name] = {
            'weights' : data['weights'].tolist(),   # int8 list
            'biases'  : data['biases'].tolist(),
            'w_shape' : data['w_shape'],
            'b_shape' : data['b_shape'],
            'dtype'   : 'int8'
        }
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(export, f, indent=2)
    print(f'✅ INT8 Weights JSON : {filepath}')


# ── Lưu Weights INT8 → Excel ─────────────────────────────────────
def save_int8_weights_excel(int8_data, filepath):
    """Mỗi layer → 2 sheet: _Weights và _Biases, dtype int8."""
    with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
        for name, data in int8_data.items():
            w = data['weights']
            b = data['biases']
            # Reshape về 2D để Excel hiển thị dễ
            pd.DataFrame(
                w.reshape(-1, w.shape[-1]).astype(int)
            ).to_excel(writer, sheet_name=f'{name}_W')
            pd.DataFrame(
                b.reshape(-1, 1).astype(int)
            ).to_excel(writer, sheet_name=f'{name}_B')
    print(f'✅ INT8 Weights Excel: {filepath}')


hw_dir = os.path.join(OUTPUT_DIR, 'hardware')
save_int8_weights_json(int8_data,  os.path.join(hw_dir, 'weights_int8.json'))
save_int8_weights_excel(int8_data, os.path.join(hw_dir, 'weights_int8.xlsx'))

In [ ]:
# ── Lưu Shift Parameters → JSON ──────────────────────────────────
#
# shift_params.json chứa SHIFT_BITS cho từng layer.
# Hardware dùng để tính: acc_int8 = clip(acc_int32 >> SHIFT_BITS, -127, 127)
#
# Giải thích cách tính SHIFT_BITS tự động:
#   - Sau MAC: acc_int32 = Σ(x_int8 × w_int8)
#   - Giá trị tối đa lý thuyết = 127 × 127 × N_inputs
#   - Số bit cần shift = floor(log2(127 × N_inputs))
#   - Nhưng dùng SHIFT_BITS cố định từ config cũng hợp lệ nếu bạn
#     muốn đơn giản hóa hardware (tránh logic shift khác nhau mỗi layer)

def compute_shift_bits(layer_name, int8_data, global_shift=SHIFT_BITS):
    """Tính shift bits cho layer dựa theo số lượng input connections."""
    w = int8_data[layer_name]['weights']
    # N_inputs = số phần tử trên mỗi output neuron
    n_inputs = int(np.prod(w.shape[:-1]))
    # Shift tự động: đủ để đưa max accumulator về ~127
    auto_shift = max(0, int(np.floor(np.log2(127 * n_inputs + 1e-8))) - 7)
    return auto_shift if auto_shift > 0 else global_shift


shift_params = {}
layer_meta   = []

for name, data in int8_data.items():
    sb = compute_shift_bits(name, int8_data)
    shift_params[name] = {
        'shift_bits' : sb,
        'w_shape'    : data['w_shape'],
        'b_shape'    : data['b_shape'],
        'n_inputs'   : int(np.prod(data['weights'].shape[:-1])),
        'note'       : f'acc_int32 >> {sb} → clip(-127,127) → INT8'
    }
    layer_meta.append({
        'Layer'       : name,
        'W Shape'     : str(data['w_shape']),
        'N inputs'    : int(np.prod(data['weights'].shape[:-1])),
        'Shift Bits'  : sb,
        'Hardware op' : f'acc >> {sb}',
        'W min/max'   : f"{data['weights'].min()} / {data['weights'].max()}",
        'B min/max'   : f"{data['biases'].min()} / {data['biases'].max()}",
    })
    print(f'  {name:20s} | shift={sb} | n_inputs={int(np.prod(data["weights"].shape[:-1]))}')

# Lưu shift_params.json
sp_path = os.path.join(hw_dir, 'shift_params.json')
with open(sp_path, 'w', encoding='utf-8') as f:
    json.dump(shift_params, f, indent=2)
print(f'\n✅ shift_params.json : {sp_path}')

# Append sheet vào weights_int8.xlsx
excel_path = os.path.join(hw_dir, 'weights_int8.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    pd.DataFrame(layer_meta).to_excel(writer, sheet_name='Shift_Params', index=False)
print(f'✅ Sheet Shift_Params → {excel_path}')

## 🏗️ 14. Xuất Kiến Trúc Mô Hình

In [ ]:
arch_info = []
for i, layer in enumerate(qat_model.layers):
    try:
        in_shape  = str(layer.input_shape)
        out_shape = str(layer.output_shape)
    except Exception:
        in_shape = out_shape = 'N/A'

    is_qat = isinstance(layer, (QATConv2D, QATDense))
    arch_info.append({
        'No.'         : i,
        'Layer Name'  : layer.name,
        'Layer Type'  : layer.__class__.__name__,
        'Input Shape' : in_shape,
        'Output Shape': out_shape,
        'Total Params': layer.count_params(),
        'QAT Layer'   : '✓' if is_qat else '',
        'W dtype'     : 'INT8' if is_qat else 'N/A',
    })

df_arch = pd.DataFrame(arch_info)

# Lưu JSON
arch_json_path = os.path.join(hw_dir, 'model_architecture.json')
df_arch.to_json(arch_json_path, orient='records', indent=2, force_ascii=False)
print(f'✅ Architecture JSON : {arch_json_path}')

# Append sheet vào Excel
with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_arch.to_excel(writer, sheet_name='Architecture', index=False)
print(f'✅ Sheet Architecture → {excel_path}')

print('\n')
print(df_arch.to_string(index=False))

## ✅ 15. Tổng Kết Output

Toàn bộ file xuất ra nằm trong thư mục `OUTPUT_DIR/hardware/`:

In [ ]:
print('=' * 60)
print('TỔNG KẾT OUTPUT')
print('=' * 60)

file_desc = {
    'hardware/weights_int8.json' : 'Weights INT8 thuần túy → nạp lên hardware ROM',
    'hardware/weights_int8.xlsx' : 'Weights INT8 + Shift Params + Architecture (Excel)',
    'hardware/shift_params.json' : 'SHIFT_BITS từng layer → hardcode vào datapath',
    'hardware/model_architecture.json': 'Kiến trúc layer → thiết kế datapath hardware',
    'cache/dataset.npz'          : 'Cache dataset → lần sau bỏ qua bước đọc ảnh',
    'checkpoint_qat.weights.h5'  : 'Checkpoint QAT → load lại để re-export',
    'training_history.png'       : 'Biểu đồ quá trình huấn luyện',
    'confusion_matrix.png'       : 'Confusion matrix',
}

for fname, desc in file_desc.items():
    full = os.path.join(OUTPUT_DIR, fname)
    status = '✅' if os.path.exists(full) else '❌'
    size   = f'{os.path.getsize(full)/1024:.1f} KB' if os.path.exists(full) else 'missing'
    print(f'{status}  {fname:<40s}  [{size:>10s}]  {desc}')

print('\n' + '=' * 60)
print('HƯỚNG DẪN SỬ DỤNG TRÊN HARDWARE')
print('=' * 60)
print('''
1. Load weights_int8.json → nạp vào ROM/BRAM
2. Load shift_params.json → hardcode SHIFT_BITS vào datapath

3. Inference flow (mỗi layer Conv/Dense):
   acc_int32 = MAC(input_int8, weight_int8 + bias_int8)
   acc_int8  = clip(acc_int32 >> SHIFT_BITS, -127, 127)
   if relu_layer:
       acc_int8 = clip(acc_int8, 0, 127)

4. Output layer Dense(2) → giữ INT32 thô:
   if out[0] > out[1]: → Benign
   if out[1] > out[0]: → Malware
   (không cần softmax, không cần float)
''')
print('=' * 60)